# Ebbinghaus kill test v2 (Qwen2-VL-7B)
Runtime: T4 GPU. Run cells top to bottom. The model-loading cell is already done if your session is still alive; if you restart the session, re-run everything from the top.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.1 MB/s eta 0:00:00


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount failed, using local disk instead:', e)

Mounted at /content/drive


# Ebbinghaus kill test (Qwen2-VL-7B)

 1. Every image is scored TOGETHER WITH ITS MIRROR IMAGE. The model has a strong "left" bias
    (about 88% "left" for equal circles), which would otherwise dominate the PSE. With
    f = (s_orig - s_mirror) / 2 any additive position/word bias cancels exactly.
 2. Two prompts: "plain" (just asks which circle is larger) and "target" (explicitly says to
    ignore the grey circles). In v1 the large inducers seemed to make the model judge overall
    cluster size, which pushes the opposite way to the human illusion. Reporting both prompts
    shows how much of the effect is the model attending to the surround vs. the target disc.
 3. Fewer seeds (10) because each image is now scored 4 times (2 prompts x 2 mirrors).

Sign convention: x = log(radius_left / radius_right).
  Condition "A": SMALL inducers around the LEFT target, LARGE around the RIGHT.
                 A human-like illusion makes the left look larger, so the left must be physically
                 SMALLER to look equal  ->  PSE_A < 0.
  Condition "B": mirror image of that  ->  PSE_B > 0.
  Human-like illusion  =>  delta = PSE_A - PSE_B < 0.   delta > 0 = opposite (e.g. cluster-size effect).

Colab (T4): paste each `# %%` block into its own cell, in order. Run the calibration cell
(Cell 6) first, then the full run.

In [3]:
# Cell 1: config

import os, json, math, itertools, random
import numpy as np

CFG = dict(
    out_dir=("/content/drive/MyDrive/vlm_psychophysics/ebbinghaus_pilot"
             if os.path.isdir("/content/drive/MyDrive") else "/content/ebbinghaus_pilot"),
    results_name="results_v2.jsonl",
    n_seeds=10,
    n_ratios=13,
    x_max=0.30,                   # log-ratio range is [-x_max, +x_max]
    gaps=[6],                     # extra distance between target and inducers (px)
    conditions=["A", "B", "equal", "none"],
    canvas=(768, 384),
    r_ref=30,
    n_inducers=(6, 8),
    model_id="Qwen/Qwen2-VL-7B-Instruct",
)

In [4]:
# Cell 2: stimulus generator + pixel-level verification

from PIL import Image, ImageDraw, ImageOps
from scipy import ndimage

R_SMALL, R_LARGE, R_MED = 10, 28, 18
TARGET_MAX = 42
TARGET_COLORS = [(214, 39, 40), (31, 119, 180), (44, 160, 44)]
INDUCER_COLOR = (128, 128, 128)


def _ring_distance(r_ind, gap):
    """Fixed centre distance so inducers never touch the largest possible target."""
    return TARGET_MAX + gap + r_ind


def _draw_disc(draw, c, r, color):
    draw.ellipse([c[0] - r, c[1] - r, c[0] + r, c[1] + r], fill=color)


def make_image(cond, x, gap, seed, cfg=CFG, mirror=False):
    """Return (PIL image, metadata). x = log(r_left / r_right) of the UNMIRRORED image.
    mirror=True flips the finished image left-right (so the 'left' circle is the original right one)."""

    rng = random.Random(f"layout-{seed}-{gap}")
    W, H = cfg["canvas"]
    r_right = cfg["r_ref"]
    r_left = r_right * math.exp(x)
    cx_l, cx_r = W // 4, 3 * W // 4
    cy = H // 2 + rng.randint(-8, 8)
    centers = {"left": (cx_l, cy), "right": (cx_r, cy)}
    t_color = rng.choice(TARGET_COLORS)

    img = Image.new("RGB", (W, H), (255, 255, 255))
    d = ImageDraw.Draw(img)

    inducer = {"A": {"left": R_SMALL, "right": R_LARGE},
               "B": {"left": R_LARGE, "right": R_SMALL},
               "equal": {"left": R_MED, "right": R_MED},
               "none": {"left": None, "right": None}}[cond]
    n_ind = rng.randint(*cfg["n_inducers"])
    meta_ind = {}
    for side in ("left", "right"):
        r_ind = inducer[side]
        phase = rng.uniform(0, 2 * math.pi)
        if r_ind is None:
            continue
        D = _ring_distance(r_ind, gap)
        meta_ind[side] = dict(r=r_ind, D=D, n=n_ind, phase=phase)
        assert 2 * D * math.sin(math.pi / n_ind) > 2 * r_ind + 1, "inducers overlap; lower n or radius"
        for k in range(n_ind):
            a = phase + 2 * math.pi * k / n_ind
            c = (centers[side][0] + D * math.cos(a), centers[side][1] + D * math.sin(a))
            _draw_disc(d, c, r_ind, INDUCER_COLOR)
    _draw_disc(d, centers["left"], r_left, t_color)
    _draw_disc(d, centers["right"], r_right, t_color)

    if mirror:
        img = ImageOps.mirror(img)
    meta = dict(cond=cond, x=float(x), gap=gap, seed=seed, r_left=r_left, r_right=r_right, mirror=mirror,
                n_ind=n_ind, inducers=meta_ind, target_color=t_color, cy=cy)
    return img, meta


def measure_targets(img, color):
    """Measure the two target radii from pixels (area -> radius). Returns (r_left, r_right) as seen in the image."""
    a = np.asarray(img)
    mask = np.all(a == np.array(color), axis=-1)
    lab, n = ndimage.label(mask)
    assert n == 2, f"expected 2 target blobs, found {n}"
    areas = ndimage.sum(mask, lab, index=[1, 2])
    coms = ndimage.center_of_mass(mask, lab, index=[1, 2])
    order = np.argsort([c[1] for c in coms])
    return tuple(math.sqrt(areas[i] / math.pi) for i in order)


def verify(img, meta, tol=1.5):
    rl, rr = measure_targets(img, meta["target_color"])
    exp_l, exp_r = (meta["r_right"], meta["r_left"]) if meta["mirror"] else (meta["r_left"], meta["r_right"])
    assert abs(rl - exp_l) < tol and abs(rr - exp_r) < tol, (rl, rr, exp_l, exp_r)
    return rl, rr


def build_design(cfg=CFG):
    xs = np.linspace(-cfg["x_max"], cfg["x_max"], cfg["n_ratios"])
    return [dict(cond=c, gap=g, x=float(x), seed=s)
            for c, g, x, s in itertools.product(cfg["conditions"], cfg["gaps"], xs, range(cfg["n_seeds"]))]

In [5]:
# Cell 3: analysis (PSE per condition, delta, bootstrap CI, plot). .

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def fit_pse(x, s):
    """s = bias-free log-odds that the LEFT circle is larger. Fit s = a + b*x. PSE = -a/b, JND(75%) = ln3 / b."""
    b, a = np.polyfit(x, s, 1)
    return dict(PSE=-a / b, slope=b, JND=math.log(3) / abs(b), a=a)


def pse_table(df, col="score", by=("cond", "gap")):
    out = []
    for key, g in df.groupby(list(by)):
        r = fit_pse(g["x"].values, g[col].values)
        out.append(dict(zip(by, key if isinstance(key, tuple) else (key,)), **r))
    return pd.DataFrame(out)


def delta_ab(df, gap=None, col="score"):
    d = df if gap is None else df[df["gap"] == gap]
    pa = fit_pse(d[d.cond == "A"]["x"].values, d[d.cond == "A"][col].values)["PSE"]
    pb = fit_pse(d[d.cond == "B"]["x"].values, d[d.cond == "B"][col].values)["PSE"]
    return pa - pb, pa, pb


def bootstrap_delta(df, gap=None, col="score", n_boot=2000, seed=0):
    """Resample LAYOUT SEEDS (not rows) with replacement; the unit of independence is the layout."""
    rng = np.random.default_rng(seed)
    d = df if gap is None else df[df["gap"] == gap]
    seeds = d["seed"].unique()
    groups = {s: d[d.seed == s] for s in seeds}
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(seeds, size=len(seeds), replace=True)
        boot = pd.concat([groups[s] for s in pick], ignore_index=True)
        try:
            vals.append(delta_ab(boot, col=col)[0])
        except Exception:
            continue
    vals = np.array(vals)
    return dict(delta=delta_ab(d, col=col)[0], lo=np.percentile(vals, 2.5), hi=np.percentile(vals, 97.5), n=len(vals))


def sanity_easy_ends(df, col="score", frac=0.25):
    """In the 'none' condition the extreme ratios are unambiguous; the bias-free score should have the right sign."""
    d = df[df.cond == "none"]
    lo, hi = d["x"].quantile(frac), d["x"].quantile(1 - frac)
    return dict(acc_left_smaller=float((d[d.x <= lo][col] < 0).mean()),
                acc_left_larger=float((d[d.x >= hi][col] > 0).mean()))


def plot_curves(df, gap, path, col="score", title_extra=""):
    colors = {"A": "#d95f02", "B": "#1b9e77", "equal": "#7570b3", "none": "#666666"}
    fig, ax = plt.subplots(figsize=(7, 4.5))
    xs = np.linspace(df.x.min(), df.x.max(), 200)
    d = df[df.gap == gap]
    for cond in ["A", "B", "equal", "none"]:
        g = d[d.cond == cond]
        if g.empty:
            continue
        m = g.groupby("x")[col].mean()
        ax.plot(m.index, 1 / (1 + np.exp(-m.values)), "o", color=colors[cond], ms=4, alpha=.8)
        f = fit_pse(g["x"].values, g[col].values)
        ax.plot(xs, 1 / (1 + np.exp(-(f["a"] + f["slope"] * xs))), "-", color=colors[cond],
                label=f"{cond}: PSE = {f['PSE']:+.3f}")
        ax.axvline(f["PSE"], color=colors[cond], lw=.7, ls=":")
    ax.axhline(.5, color="k", lw=.5)
    ax.set_xlabel("true log size ratio, log(r_left / r_right)")
    ax.set_ylabel("P('left is larger'), bias-free")
    ax.set_title(f"Ebbinghaus, gap = {gap}px {title_extra}", fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

In [6]:
# Cell 4: load Qwen2-VL-7B in 4-bit

def load_qwen(cfg=CFG):
    import torch
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=False)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        cfg["model_id"], quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
    model.eval()
    proc = AutoProcessor.from_pretrained(cfg["model_id"])
    return model, proc

In [7]:
# Cell 5: scoring. s = log P('left') - log P('right') at the first answer token

PROMPTS = {
    "plain": "Which circle is larger, the one on the left or the one on the right? Answer with one word: left or right.",
    "target": "Ignore the grey circles. Which of the two coloured circles is larger, the one on the left "
              "or the one on the right? Answer with one word: left or right.",
}


def answer_token_ids(tok):
    def ids(words):
        out = []
        for w in words:
            t = tok.encode(w, add_special_tokens=False)
            if t:
                out.append(t[0])
        return sorted(set(out))
    return ids(["left", "Left", " left", " Left"]), ids(["right", "Right", " right", " Right"])


def logodds_left(model, proc, img, prompt, left_ids, right_ids):
    import torch
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[text], images=[img], return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    lp = torch.log_softmax(logits, dim=-1)
    return (torch.logsumexp(lp[left_ids], 0) - torch.logsumexp(lp[right_ids], 0)).item()


def run_scoring(model, proc, cfg=CFG, limit=None, verify_images=True):
    """Resumable. For each design row scores the image AND its mirror, for each prompt.
    Saved per prompt: score_<name> = (s_orig - s_mirror)/2  (bias-free), bias_<name> = (s_orig + s_mirror)/2."""
    os.makedirs(cfg["out_dir"], exist_ok=True)
    path = os.path.join(cfg["out_dir"], cfg["results_name"])
    done = set()
    if os.path.exists(path):
        for line in open(path):
            r = json.loads(line)
            done.add((r["cond"], r["gap"], round(r["x"], 6), r["seed"]))
    left_ids, right_ids = answer_token_ids(proc.tokenizer)
    print("left ids:", left_ids, [proc.tokenizer.decode([i]) for i in left_ids])
    print("right ids:", right_ids, [proc.tokenizer.decode([i]) for i in right_ids])
    todo = [r for r in build_design(cfg) if (r["cond"], r["gap"], round(r["x"], 6), r["seed"]) not in done]
    if limit:
        todo = todo[:limit]
    with open(path, "a") as f:
        for i, row in enumerate(todo):
            img, meta = make_image(row["cond"], row["x"], row["gap"], row["seed"], cfg, mirror=False)
            img_m, meta_m = make_image(row["cond"], row["x"], row["gap"], row["seed"], cfg, mirror=True)
            if verify_images:
                verify(img, meta); verify(img_m, meta_m)
            rec = dict(row)
            for name, prompt in PROMPTS.items():
                so = logodds_left(model, proc, img, prompt, left_ids, right_ids)
                sm = logodds_left(model, proc, img_m, prompt, left_ids, right_ids)
                rec[f"score_{name}"] = (so - sm) / 2
                rec[f"bias_{name}"] = (so + sm) / 2
                rec[f"raw_{name}"] = [so, sm]
            f.write(json.dumps(rec) + "\n")
            f.flush()
            if i % 25 == 0:
                print(f"{i}/{len(todo)}  " + "  ".join(f"{n}: {rec[f'score_{n}']:+.2f}" for n in PROMPTS))
    return path

In [8]:
# Cell 6: calibration and analysis of the full run

def run_calibration(model, proc, cfg=CFG):
    """No-inducer images over a wide range, mirrored. Checks: bias-free curve crosses 0 near x=0 and is odd-symmetric."""
    cal = dict(cfg)
    cal.update(out_dir=cfg["out_dir"] + "_calib", results_name="calib_v2.jsonl",
               conditions=["none"], n_seeds=5, n_ratios=9, x_max=1.0, gaps=[6])
    path = run_scoring(model, proc, cfg=cal)
    c = pd.read_json(path, lines=True)
    for name in PROMPTS:
        print(f"\n--- prompt: {name} ---")
        t = c.groupby("x").agg(score=(f"score_{name}", "mean"), bias=(f"bias_{name}", "mean")).round(2)
        print(t)
        print({k: round(v, 3) for k, v in fit_pse(c.x.values, c[f"score_{name}"].values).items() if k in ("PSE", "slope")})
    return c


def analyse(path, cfg=CFG):
    df = pd.read_json(path, lines=True)
    print("rows:", len(df))
    for name in PROMPTS:
        col = f"score_{name}"
        print(f"\n===== prompt: {name} =====")
        print("sanity (want near 1.0):", sanity_easy_ends(df, col))
        print(pse_table(df, col).round(3))
        for gap in sorted(df.gap.unique()):
            print(f"gap={gap}  delta = PSE_A - PSE_B (negative = human-like):",
                  {k: (round(v, 3) if k != 'n' else v) for k, v in bootstrap_delta(df, gap=gap, col=col).items()})
            plot_curves(df, gap, os.path.join(cfg["out_dir"], f"psychometric_{name}_gap{gap}.png"), col=col, title_extra=f"({name} prompt)")
    return df

In [9]:
model, proc = load_qwen()
cal = run_calibration(model, proc)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

left ids: [2115, 2359, 5415, 13727] [' left', 'left', 'Left', ' Left']
right ids: [1290, 1291, 5979, 10083] [' right', 'right', 'Right', ' Right']
0/45  plain: -2.91  target: -2.54
25/45  plain: +1.79  target: +1.22

--- prompt: plain ---
       score  bias
x                 
-1.00  -2.87  1.67
-0.75  -2.83  1.65
-0.50  -2.55  1.71
-0.25  -1.30  2.39
 0.00   0.03  3.03
 0.25   2.04  1.93
 0.50   3.51  1.55
 0.75   4.04  1.44
 1.00   4.42  1.45
{'PSE': np.float64(-0.115), 'slope': np.float64(4.351)}

--- prompt: target ---
       score  bias
x                 
-1.00  -2.44  1.91
-0.75  -2.33  1.94
-0.50  -2.01  2.04
-0.25  -0.87  2.62
 0.00   0.02  3.00
 0.25   1.36  2.35
 0.50   3.07  1.80
 0.75   3.49  1.85
 1.00   3.74  1.91
{'PSE': np.float64(-0.123), 'slope': np.float64(3.637)}


In [11]:
# # Full run: 520 images x 2 prompts x 2 mirrors.
path = run_scoring(model, proc)
df = analyse(path)

left ids: [2115, 2359, 5415, 13727] [' left', 'left', 'Left', ' Left']
right ids: [1290, 1291, 5979, 10083] [' right', 'right', 'Right', ' Right']
0/514  plain: -2.28  target: -0.66
25/514  plain: -2.60  target: -0.39
50/514  plain: -1.74  target: -0.46
75/514  plain: -1.95  target: -0.05
100/514  plain: -0.51  target: +0.26
125/514  plain: +1.81  target: -0.14
150/514  plain: +1.04  target: +0.06
175/514  plain: +2.27  target: +0.08
200/514  plain: +1.64  target: +0.32
225/514  plain: +2.32  target: +0.14
250/514  plain: +2.32  target: +0.72
275/514  plain: -0.69  target: -0.29
300/514  plain: -0.53  target: -0.24
325/514  plain: +0.08  target: +0.08
350/514  plain: +0.61  target: +0.22
375/514  plain: +1.73  target: +0.57
400/514  plain: -1.16  target: -0.73
425/514  plain: -0.41  target: -0.20
450/514  plain: -0.05  target: -0.02
475/514  plain: +0.50  target: +0.29
500/514  plain: +1.95  target: +1.32
rows: 520

===== prompt: plain =====
sanity (want near 1.0): {'acc_left_smaller':